# ML-09 — Validation and Research Claim Audit

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Two paper findings + my methodology questions

*Pick two findings from the FlyRank research paper. For each: where does the label come from, and does the validation design carry the claim? Constructive tone.*

**I picked:**
- Finding A: "The Freshness Multiplier — 365+ day content refreshed within 30 days shows large health gains."
- Finding B: "The Content Performance Curve — content peaks at 61–90 days and declines after 270 days."

For each I ask concrete, review-style methodology questions and suggest small verification queries.

Finding A — Freshness Multiplier (where label comes from)
- Where the label/value comes from
  - The paper compares growth ratios by freshness bucket; the "boost" appears to be a ratio of health/impression metrics pre- vs post-refresh or refreshed vs not-refreshed.
- Methodology questions I would ask (and quick checks)
  1. Refresh definition: Is "refreshed within 30 days" taken from CMS timestamps or from diffs/major-edit detection? (Check: count pages with small vs large timestamp deltas; show bucket sizes.)
  2. Sample size & stability: How many pages are in the 365+ bucket and in the "refreshed" subbucket? (Check: report n per cell; if tiny, the ratio is unstable.)
  3. Reverse causality: Are high-health pages more likely to be refreshed (teams invest in winners)? (Check: distribution of pre-refresh health for refreshed vs non-refreshed.)
  4. Topic & visibility confounding: Are certain topics or high-impr pages overrepresented among refreshed pages? (Check: topic/intent or impressions distribution by refreshed flag.)
  5. Measurement window alignment: If "refreshed within 30 days" overlaps the 90-day label window, do features/labels overlap? (Check: ensure features used for prediction pre-date the refresh event.)
- Small verification queries to run:
  - Show counts per (age_bucket × refreshed_flag) and the mean/median impressions & health pre-refresh.
  - If refreshed pages have much higher pre-refresh health, highlight reverse-causality risk.

Finding B — Content Performance Curve (where label comes from)
- Where the label/value comes from
  - The paper computes health or performance summaries by age buckets (e.g., 0–30, 31–60, 61–90, ...). The curve is an age-bucket vs mean health plot.
- Methodology questions I would ask (and quick checks)
  1. Age-bucket choice: Why these exact ranges (61–90)? Are bucket sizes similar or heavily imbalanced? (Check: counts per bucket.)
  2. Survivor bias: Are older pages only the high-performing survivors? (Check: percent of pages in each age bucket that have non-zero impressions / were indexed.)
  3. Health circularity: Is the health score computed using signals that also depend on age (e.g., impressions aggregated over time)? (Check: health components and whether they overlap with age-based availability.)
  4. Refresh / reactivation: Do pages later refreshed move buckets and re-enter the peak region? (Check: track a small sample of pages that were refreshed.)
  5. Seasonality & cohort comparability: Are pages born at different times (different cohorts) being compared without normalizing for seasonality or index growth? (Check: cohort counts by creation month.)
- Small verification queries:
  - Show n per age bucket, mean health, median impressions, and percent with >0 impressions.
  - For older buckets, show the distribution of pre-existing traffic (to reveal survivor bias).



## 2. My model under an honest split (before/after)

*Re-run your Week-5 model under a grouped or time-aware split. Show both numbers.*

In [1]:
import numpy as np
import pandas as pd
from sklearn.model_selection import train_test_split, GroupShuffleSplit
from sklearn.ensemble import RandomForestRegressor
from sklearn.pipeline import make_pipeline
from sklearn.impute import SimpleImputer
from sklearn.preprocessing import OneHotEncoder
from sklearn.compose import ColumnTransformer
import random
import warnings
warnings.filterwarnings("ignore")

RANDOM_SEED = 42
np.random.seed(RANDOM_SEED)
random.seed(RANDOM_SEED)

RAW_URL = "https://raw.githubusercontent.com/reezcon/First-ML-Pipeline/main/data/raw/content_refresh_anonymized.csv"
df = pd.read_csv(RAW_URL)
df = df.copy()

# Core target and base rate
df["engagement_rate"] = df["engagement_rate"].fillna(0).astype(float)
y_all = df["engagement_rate"]
print("Dataset rows:", len(df))
print("Overall base rate (mean engagement_rate) = {:.4f}\n".format(y_all.mean()))

# Baseline rule (recreate simple heuristic from Week-5)
df["impressions_90d"] = pd.to_numeric(df.get("impressions_90d", 0), errors="coerce").fillna(0)
df["ctr"] = pd.to_numeric(df.get("ctr"), errors="coerce") if "ctr" in df.columns else None
df["avg_position"] = pd.to_numeric(df.get("avg_position"), errors="coerce") if "avg_position" in df.columns else None
df["avg_position_missing"] = ((df["avg_position"].isna()) | (df["avg_position"] == 0)).astype(int)

TH_IMPR_HIGH = 1000
TH_IMPR_MED = 500
TH_CTR_LOW = 0.5
TH_POSITION_POOR = 10

df["high_impr"] = (df["impressions_90d"] >= TH_IMPR_HIGH).astype(int)
df["med_impr"] = ((df["impressions_90d"] >= TH_IMPR_MED) & (df["impressions_90d"] < TH_IMPR_HIGH)).astype(int)
if df["ctr"] is not None:
    df["low_ctr"] = ((df["ctr"].notna()) & (df["ctr"] <= TH_CTR_LOW)).astype(int)
else:
    df["low_ctr"] = 0
df["poor_position"] = ((df["avg_position"].notna()) & (df["avg_position"] > TH_POSITION_POOR) & (df["avg_position_missing"] == 0)).astype(int)
df["missing_position"] = df["avg_position_missing"].astype(int)

df["baseline_score"] = 3*df["high_impr"] + 2*df["med_impr"] + 2*df["low_ctr"] + 2*df["poor_position"] + 1*df["missing_position"]

# Shared features used for model (same as Week-5)
features = ["content_type", "position_tier", "freshness_tier", "word_count", "competition_level", "cpc", "search_volume"]
X = df[features].copy()
y = df["engagement_rate"].astype(float)

# Preprocessing pipeline (categorical OHE, numeric median impute)
categorical_cols = ["content_type", "position_tier", "freshness_tier", "competition_level"]
numeric_cols = ["word_count", "cpc", "search_volume"]

cat_pipe = make_pipeline(SimpleImputer(strategy="constant", fill_value="MISSING"), OneHotEncoder(handle_unknown="ignore"))
num_pipe = make_pipeline(SimpleImputer(strategy="median"))

pre = ColumnTransformer([
    ("cat", cat_pipe, categorical_cols),
    ("num", num_pipe, numeric_cols),
], remainder="drop")

def train_and_eval(train_idx, test_idx, desc="run"):
    X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
    y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]
    df_test = df.iloc[test_idx].reset_index(drop=True)
    model = make_pipeline(pre, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    model.fit(X_train, y_train)
    preds_test = model.predict(X_test)

    # Helper: mean engagement in top K by score
    def mean_top_k_score(score_array, labels, k):
        order = np.argsort(-np.asarray(score_array))
        topk = order[:k]
        return np.asarray(labels).astype(float)[topk].mean()

    # Baseline on this test fold
    baseline_scores_test = df_test["baseline_score"].values
    base_test = y_test.mean()

    results = {"base_rate_test": base_test}
    for k in (20, 50, 100):
        m_baseline = mean_top_k_score(baseline_scores_test, df_test["engagement_rate"].values, k)
        m_model = mean_top_k_score(preds_test, df_test["engagement_rate"].values, k)
        results[f"top_{k}_baseline"] = m_baseline
        results[f"top_{k}_model"] = m_model

    # Return model and test diagnostics for later inspection
    df_test = df_test.copy()
    df_test["pred_engagement"] = preds_test
    return model, results, df_test

# 1) BEFORE: random split (representative but optimistic if groups repeat)
train_idx_r, test_idx_r = train_test_split(np.arange(len(X)), test_size=0.2, random_state=RANDOM_SEED)
model_r, results_r, df_test_r = train_and_eval(train_idx_r, test_idx_r, desc="random split")

print("=== BEFORE: random split ===")
print("Base rate on TEST (random) = {:.4f}".format(results_r["base_rate_test"]))
print("Mean engagement in top-20 — baseline: {:.4f} | model: {:.4f}".format(results_r["top_20_baseline"], results_r["top_20_model"]))
print("Mean engagement in top-50 — baseline: {:.4f} | model: {:.4f}".format(results_r["top_50_baseline"], results_r["top_50_model"]))
print("Mean engagement in top-100 — baseline: {:.4f} | model: {:.4f}\n".format(results_r["top_100_baseline"], results_r["top_100_model"]))

# 2) HONEST: grouped-by-client split
groups = df["client_id"].fillna("MISSING_CLIENT")
gss = GroupShuffleSplit(n_splits=1, test_size=0.2, random_state=RANDOM_SEED)
train_idx_g, test_idx_g = next(gss.split(X, y, groups=groups))
model_g, results_g, df_test_g = train_and_eval(train_idx_g, test_idx_g, desc="grouped split")

print("=== AFTER: grouped-by-client split (honest) ===")
print("Base rate on TEST (grouped) = {:.4f}".format(results_g["base_rate_test"]))
print("Mean engagement in top-20 — baseline: {:.4f} | model: {:.4f}".format(results_g["top_20_baseline"], results_g["top_20_model"]))
print("Mean engagement in top-50 — baseline: {:.4f} | model: {:.4f}".format(results_g["top_50_baseline"], results_g["top_50_model"]))
print("Mean engagement in top-100 — baseline: {:.4f} | model: {:.4f}\n".format(results_g["top_100_baseline"], results_g["top_100_model"]))

# 3) Synthetic leak test: create an explicit leaky feature (label + small noise), retrain under grouped split,
#    see the jump toward perfect predictions, then remove it again (sanity check).
print("=== Synthetic leak verification (sanity check) ===")
df_leak = df.copy()
# Create a leaky numeric feature that is nearly the label (we add small noise)
df_leak["synthetic_leak"] = df_leak["engagement_rate"] + np.random.normal(scale=1e-2, size=len(df_leak))

# Build X_leak with the synthetic leak appended as numeric
features_leak = features + ["synthetic_leak"]
X_leak = df_leak[features_leak].copy()
# Preprocessing must handle the new numeric column; update pipelines:
numeric_cols_leak = numeric_cols + ["synthetic_leak"]
pre_leak = ColumnTransformer([
    ("cat", cat_pipe, categorical_cols),
    ("num", num_pipe, numeric_cols_leak),
], remainder="drop")

def train_and_eval_with_pre(preprocessor, train_idx, test_idx):
    X_train = pd.concat([X_leak.iloc[train_idx][categorical_cols], X_leak.iloc[train_idx][numeric_cols_leak]], axis=1)
    X_test = pd.concat([X_leak.iloc[test_idx][categorical_cols], X_leak.iloc[test_idx][numeric_cols_leak]], axis=1)
    model = make_pipeline(preprocessor, RandomForestRegressor(n_estimators=200, random_state=RANDOM_SEED, n_jobs=-1))
    model.fit(X_train, y.iloc[train_idx])
    preds_test = model.predict(X_test)
    df_test_local = df_leak.iloc[test_idx].reset_index(drop=True)
    df_test_local["pred_engagement"] = preds_test
    # top-k helper
    def mean_top_k_score(score_array, labels, k):
        order = np.argsort(-np.asarray(score_array))
        topk = order[:k]
        return np.asarray(labels).astype(float)[topk].mean()
    results = {}
    for k in (20, 50, 100):
        results[f"top_{k}_model"] = mean_top_k_score(preds_test, df_test_local["engagement_rate"].values, k)
    results["base_rate_test"] = y.iloc[test_idx].mean()
    return model, results

# Train with synthetic leak under grouped split
model_leak, results_leak = train_and_eval_with_pre(pre_leak, train_idx_g, test_idx_g)
print("With synthetic leaky feature included (grouped test):")
print("Base rate on TEST = {:.4f}".format(results_leak["base_rate_test"]))
print("Mean engagement in top-20 — model (leak): {:.4f}".format(results_leak["top_20_model"]))
print("Mean engagement in top-50 — model (leak): {:.4f}".format(results_leak["top_50_model"]))
print("Mean engagement in top-100 — model (leak): {:.4f}\n".format(results_leak["top_100_model"]))

print("Sanity check passed if the 'leak' model leaps much higher than the honest model above. Remove synthetic_leak and keep honest numbers.")

Dataset rows: 30000
Overall base rate (mean engagement_rate) = 2.5345

=== BEFORE: random split ===
Base rate on TEST (random) = 2.5170
Mean engagement in top-20 — baseline: 3.7020 | model: 0.3675
Mean engagement in top-50 — baseline: 2.8554 | model: 2.0428
Mean engagement in top-100 — baseline: 2.2822 | model: 2.1528

=== AFTER: grouped-by-client split (honest) ===
Base rate on TEST (grouped) = 2.9117
Mean engagement in top-20 — baseline: 2.3415 | model: 0.3780
Mean engagement in top-50 — baseline: 2.9996 | model: 1.5798
Mean engagement in top-100 — baseline: 3.6169 | model: 2.5390

=== Synthetic leak verification (sanity check) ===
With synthetic leaky feature included (grouped test):
Base rate on TEST = 2.9117
Mean engagement in top-20 — model (leak): 87.5005
Mean engagement in top-50 — model (leak): 65.0002
Mean engagement in top-100 — model (leak): 50.8442

Sanity check passed if the 'leak' model leaps much higher than the honest model above. Remove synthetic_leak and keep honest 

## 3. Leakage audit

*The same hunt from Week 3, on your final feature set.*

## 4. Claim rewrite

*Take your own boldest sentence and rewrite it in safe language: observed, measured, directional, decision-support.*

## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.